# Findings — What Fine-Tuning Actually Buys (and what the oracle hides)

**Project:** Can LoRA-adapted SAM / MedSAM match specialist polyp networks, and generalize better to unseen datasets?

This notebook *illustrates* the results from `05_benchmark.ipynb` with the examples that were actually
evaluated. It is built around one reading rule:

> **There are two tiers of models, and they are NOT doing the same task.**
>
> - **Prompt-free (fair):** U-Net, SAM+LoRA, MedSAM+LoRA see *only the image* and must **find** the polyp.
> - **Oracle-prompted (upper bound):** vanilla SAM / MedSAM are **handed the ground-truth bounding box** —
>   they are told where the polyp is and only have to trace it.
>
> So the vanilla models' high scores are an *upper bound on boundary tracing given perfect localization*,
> **not** evidence they beat fine-tuning. The tell: their generalization gap is **negative** (they score
> higher on unseen than seen) — impossible for real generalization, and the signature of an oracle.

**The honest headline:** among the *prompt-free* models, **SAM-ViT-H + LoRA generalizes best to unseen data**
(0.792 mDice vs U-Net 0.752), with the smallest seen→unseen drop, using **0.4% of the parameters**.
What fine-tuning buys is **detection** — the thing the oracle box hands the vanilla models for free.

## 1. Setup — environment, data

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO = '/content/msu2026summer_final_project'
    if not os.path.exists(REPO):
        !git clone --quiet https://github.com/palism1/msu2026summer_final_project.git {REPO}
    %cd {REPO}
    !git fetch --quiet origin && git reset --hard origin/main
    !find . -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null; true
    !pip install -q -r requirements.txt
    !pip install -q git+https://github.com/facebookresearch/segment-anything.git

    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import gdown, shutil, zipfile
from pathlib import Path

DATA_ROOT = Path('data/polyp')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

def _dl(file_id, zip_name):
    target = DATA_ROOT / zip_name.replace('.zip', '')
    if target.exists():
        print(f'{target.name}: already present'); return
    zip_path = f'/tmp/{zip_name}'
    print(f'Downloading {target.name}...')
    gdown.download(id=file_id, output=zip_path, quiet=False, fuzzy=True)
    tmp = Path(f'/tmp/extract_{target.name}')
    tmp.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(tmp)
    extracted = [p for p in tmp.iterdir() if not p.name.startswith('__')]
    if len(extracted) == 1 and extracted[0].is_dir():
        shutil.move(str(extracted[0]), str(target))
    else:
        target.mkdir(exist_ok=True)
        for p in extracted: shutil.move(str(p), str(target / p.name))
    print(f'Done -> {target}')

_dl('1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb', 'TrainDataset.zip')
_dl('1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao', 'TestDataset.zip')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}  |  GPU: {torch.cuda.get_device_name(0) if device=="cuda" else "N/A"}')

## 2. Locate checkpoints (pulled from Drive if not local)

In [ ]:
SEED = 42
DRIVE_ROOT = Path('/content/drive/MyDrive/msu2026_checkpoints')

CKPT_MAP = {
    'unet':    Path(f'checkpoints/unet/seed{SEED}/best.pt'),
    'sam_vit_h': Path(f'checkpoints/sam_vit_h/seed{SEED}/best.pt'),
    'medsam':  Path(f'checkpoints/medsam/seed{SEED}/best.pt'),
}

for name, local_path in CKPT_MAP.items():
    if local_path.exists():
        print(f'{name:<12} found locally  -> {local_path}')
        continue
    drive_path = DRIVE_ROOT / name / f'seed{SEED}' / 'best.pt'
    if not drive_path.exists():
        raise FileNotFoundError(
            f'No checkpoint for "{name}" at {drive_path}. Train it first (train_colab.ipynb).'
        )
    local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(drive_path, local_path)
    print(f'{name:<12} pulled from Drive -> {local_path}')

## 3. Build all five models + load the test splits

Same builders as the benchmark: three trained models (U-Net, SAM+LoRA, MedSAM+LoRA) plus the two
zero-shot baselines (raw SAM / MedSAM, GT-box prompted). The zero-shot ones need no checkpoint — only
the base weights downloaded here.

In [ ]:
from src.models import build_unet, build_sam_lora, build_medsam_lora

IMG_SIZE   = 352
LORA_R     = 4
LORA_ALPHA = 8.0

models = {}

# --- U-Net ---
unet = build_unet().to(device)
unet.load_state_dict(torch.load(CKPT_MAP['unet'], map_location=device))
unet.eval()
models['U-Net (ResNet-34)'] = unet

# --- SAM ViT-H + LoRA ---
SAM_FNAME = 'sam_vit_h_4b8939.pth'
if not os.path.exists(SAM_FNAME):
    print('Downloading SAM ViT-H checkpoint (~2.4 GB)...')
    !wget -q --show-progress https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -O {SAM_FNAME}

sam_lora = build_sam_lora(
    sam_checkpoint=SAM_FNAME, model_type='vit_h',
    lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.1,
    img_size=IMG_SIZE, device=device,
).to(device)
sam_lora.load_state_dict(torch.load(CKPT_MAP['sam_vit_h'], map_location=device))
sam_lora.eval()
models['SAM-ViT-H + LoRA'] = sam_lora

# --- MedSAM + LoRA ---
import hashlib
MEDSAM_FNAME = 'medsam_vit_b.pth'
MEDSAM_URL   = 'https://zenodo.org/records/10689643/files/medsam_vit_b.pth'
MEDSAM_MD5   = '3bb6db55bd0c9ca30b61248bca72f8d6'

def _md5(path, chunk=8 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

if os.path.exists(MEDSAM_FNAME) and (os.path.getsize(MEDSAM_FNAME) < 300_000_000 or _md5(MEDSAM_FNAME) != MEDSAM_MD5):
    os.remove(MEDSAM_FNAME)
if not os.path.exists(MEDSAM_FNAME):
    print('Downloading MedSAM checkpoint (~375 MB)...')
    !wget -q --show-progress {MEDSAM_URL} -O {MEDSAM_FNAME}
assert _md5(MEDSAM_FNAME) == MEDSAM_MD5, 'MedSAM checkpoint MD5 mismatch'

medsam_lora = build_medsam_lora(
    medsam_checkpoint=MEDSAM_FNAME,
    lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.1,
    img_size=IMG_SIZE, device=device,
).to(device)
medsam_lora.load_state_dict(torch.load(CKPT_MAP['medsam'], map_location=device))
medsam_lora.eval()
models['MedSAM-ViT-B + LoRA'] = medsam_lora

# --- Zero-shot (vanilla) baselines: GT-box prompted, no training, 0 trainable params ---
from src.models import build_zeroshot_sam, build_zeroshot_medsam

ZS_PROMPT  = 'box'   # 'box' (oracle box from GT) | 'point'
ZS_BOX_PAD = 5

zeroshot_models = {}
zeroshot_models['SAM-ViT-H (zero-shot)'] = build_zeroshot_sam(
    SAM_FNAME, model_type='vit_h', device=device, prompt=ZS_PROMPT, box_padding=ZS_BOX_PAD)
zeroshot_models['MedSAM-ViT-B (zero-shot)'] = build_zeroshot_medsam(
    MEDSAM_FNAME, device=device, prompt=ZS_PROMPT, box_padding=ZS_BOX_PAD)

all_models = {**models, **zeroshot_models}
print('Trained  :', list(models.keys()))
print('Zero-shot:', list(zeroshot_models.keys()))

In [ ]:
from src.data import build_splits, get_val_transform
from src.metrics import dice_score
from src.models import is_zeroshot, box_from_mask
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage

splits = build_splits(DATA_ROOT, seed=SEED)
val_transform = get_val_transform(IMG_SIZE)

SPLIT_LABELS = {
    'seen_kvasir':   'Seen — Kvasir',
    'seen_clinicdb': 'Seen — CVC-ClinicDB',
    'cvc_colondb':   'Unseen — CVC-ColonDB',
    'etis_larib':    'Unseen — ETIS-Larib',
    'cvc_300':       'Unseen — CVC-300',
}
print('Split sizes:', {k: len(v['image_paths']) for k, v in splits.items() if k in SPLIT_LABELS})

## 4. What each model was *given*

This is the crux of the whole study. The panel below shows, for a few unseen images:
**Image** (what every model sees) · **Ground truth** · **the oracle box** drawn on the image
(what the *vanilla* SAM/MedSAM models are additionally handed).

The fine-tuned models and U-Net get **only the left image**. The vanilla models get the **red box** —
i.e. they are told where the polyp is before they segment. Keep that asymmetry in mind for every
number in this notebook.

In [ ]:
import matplotlib.patches as patches

def show_oracle_prompt(split_key, n=3, img_size=IMG_SIZE, pad=ZS_BOX_PAD):
    ips = splits[split_key]['image_paths'][:n]
    mps = splits[split_key]['mask_paths'][:n]
    fig, axes = plt.subplots(len(ips), 3, figsize=(9, 3 * len(ips)))
    if len(ips) == 1:
        axes = axes[None, :]
    for r, (imgp, maskp) in enumerate(zip(ips, mps)):
        img = np.array(PILImage.open(imgp).convert('RGB').resize((img_size, img_size)), dtype=np.uint8)
        gt = (np.array(PILImage.open(maskp).convert('L').resize((img_size, img_size), PILImage.NEAREST)) > 127).astype(np.float32)
        box = box_from_mask(gt, padding=pad)

        axes[r, 0].imshow(img)
        axes[r, 0].set_title('Image (ALL models see this)' if r == 0 else ''); axes[r, 0].axis('off')
        axes[r, 1].imshow(gt, cmap='gray')
        axes[r, 1].set_title('Ground truth' if r == 0 else ''); axes[r, 1].axis('off')
        axes[r, 2].imshow(img)
        if box is not None:
            x0, y0, x1, y1 = box
            axes[r, 2].add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                                    fill=False, edgecolor='red', linewidth=2.5))
        axes[r, 2].set_title('Oracle box handed to VANILLA only' if r == 0 else ''); axes[r, 2].axis('off')
    fig.suptitle(f'{SPLIT_LABELS[split_key]} — vanilla SAM/MedSAM are TOLD where the polyp is',
                 fontweight='bold')
    plt.tight_layout(); plt.show()

show_oracle_prompt('etis_larib', n=3)

## 5. Where the oracle box matters most

We rank unseen images by how much the box *helps* SAM: `Dice(zero-shot SAM) − Dice(SAM+LoRA)`.
The largest gaps are **detection failures** — cases where the fine-tuned model can't find the polyp
from the image alone, but the vanilla model trivially segments it once handed the box. These are the
images inflating the vanilla scores. (`scan` limits how many images are ranked, for speed.)

In [ ]:
def _bin_pred(model, img_t, img_u8, gt_t):
    if is_zeroshot(model):
        return (model.predict_prob(img_u8, gt_t) >= 0.5).astype(np.float32)
    with torch.no_grad():
        return (torch.sigmoid(model(img_t)).cpu().numpy()[0, 0] >= 0.5).astype(np.float32)

def _load_one(imgp, maskp, img_size=IMG_SIZE):
    img_pil = PILImage.open(imgp).convert('RGB')
    gt_np = (np.array(PILImage.open(maskp).convert('L')) > 127).astype(np.float32)
    aug = val_transform(image=np.array(img_pil), mask=gt_np)
    img_t = aug['image'].unsqueeze(0).to(device)
    gt_t = aug['mask'].numpy()
    img_u8 = np.array(img_pil.resize((img_size, img_size)), dtype=np.uint8)
    return img_pil, img_t, img_u8, gt_t

def rank_by_oracle_gap(split_key, scan=60):
    ft = models['SAM-ViT-H + LoRA']
    zs = zeroshot_models['SAM-ViT-H (zero-shot)']
    ips = splits[split_key]['image_paths'][:scan]
    mps = splits[split_key]['mask_paths'][:scan]
    out = []
    for i, (imgp, maskp) in enumerate(zip(ips, mps)):
        _, img_t, img_u8, gt_t = _load_one(imgp, maskp)
        d_ft = dice_score(_bin_pred(ft, img_t, img_u8, gt_t), gt_t)
        d_zs = dice_score(_bin_pred(zs, img_t, img_u8, gt_t), gt_t)
        out.append((i, d_ft, d_zs))
    out.sort(key=lambda r: r[2] - r[1], reverse=True)
    return out


def show_gallery(split_key, indices, img_size=IMG_SIZE):
    ips = [splits[split_key]['image_paths'][i] for i in indices]
    mps = [splits[split_key]['mask_paths'][i] for i in indices]
    ncol = 2 + len(all_models)
    fig, axes = plt.subplots(len(indices), ncol, figsize=(2.6 * ncol, 2.6 * len(indices)))
    if len(indices) == 1:
        axes = axes[None, :]
    for r, (imgp, maskp) in enumerate(zip(ips, mps)):
        img_pil, img_t, img_u8, gt_t = _load_one(imgp, maskp)
        axes[r, 0].imshow(img_pil.resize((img_size, img_size)))
        axes[r, 0].set_title('Image' if r == 0 else ''); axes[r, 0].axis('off')
        axes[r, 1].imshow(gt_t, cmap='gray')
        axes[r, 1].set_title('GT' if r == 0 else ''); axes[r, 1].axis('off')
        for c, (name, model) in enumerate(all_models.items(), start=2):
            pred = _bin_pred(model, img_t, img_u8, gt_t)
            d = dice_score(pred, gt_t)
            axes[r, c].imshow(pred, cmap='gray')
            tag = ('*' if is_zeroshot(model) else '')
            axes[r, c].set_title((f'{name}{tag}\n' if r == 0 else '') + f'Dice={d:.2f}', fontsize=8)
            axes[r, c].axis('off')
    fig.suptitle(f'{SPLIT_LABELS[split_key]} — biggest "oracle box helped" cases   (* = vanilla, GT-box prompted)',
                 fontweight='bold')
    plt.tight_layout(); plt.show()

gaps = rank_by_oracle_gap('etis_larib', scan=60)
top = [i for i, d_ft, d_zs in gaps[:4]]
print('Top oracle-advantage cases (idx, SAM+LoRA Dice, zero-shot Dice):')
for i, d_ft, d_zs in gaps[:4]:
    print(f'  idx {i:>3}:  fine-tuned {d_ft:.3f}   ->   oracle {d_zs:.3f}   (+{d_zs - d_ft:.3f})')
show_gallery('etis_larib', top)

## 6. The two-tier result

Blue = **prompt-free** (fair comparison). Red = **oracle-prompted** (upper bound, *not* directly
comparable). The vanilla bars are high because they were handed the box — not because they beat
fine-tuning at the actual task.

In [ ]:
# Numbers from the 05_benchmark.ipynb run (seed 42). Re-run 05_benchmark to refresh.
SUMMARY = {
    'U-Net (ResNet-34)':        dict(seen=0.9094, unseen=0.7521, gap=0.1573,  trainable=24_436_369, train_min=12.7, tier='prompt-free'),
    'SAM-ViT-H + LoRA':         dict(seen=0.8878, unseen=0.7915, gap=0.0963,  trainable=830_177,    train_min=97.0, tier='prompt-free'),
    'MedSAM-ViT-B + LoRA':      dict(seen=0.8130, unseen=0.6600, gap=0.1530,  trainable=322_273,    train_min=30.5, tier='prompt-free'),
    'SAM-ViT-H (zero-shot)':    dict(seen=0.8595, unseen=0.9052, gap=-0.0457, trainable=0,          train_min=0.0,  tier='oracle'),
    'MedSAM-ViT-B (zero-shot)': dict(seen=0.7997, unseen=0.8446, gap=-0.0449, trainable=0,          train_min=0.0,  tier='oracle'),
}

names  = list(SUMMARY)
unseen = [SUMMARY[n]['unseen'] for n in names]
colors = ['#1f77b4' if SUMMARY[n]['tier'] == 'prompt-free' else '#d62728' for n in names]
n_pf   = sum(1 for n in names if SUMMARY[n]['tier'] == 'prompt-free')

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(names, unseen, color=colors)
ax.axvspan(n_pf - 0.5, len(names) - 0.5, color='#d62728', alpha=0.06)
ax.text((n_pf + len(names) - 1) / 2, 0.06,
        'ORACLE — GT box given\n(upper bound, NOT a fair comparison)',
        ha='center', color='#d62728', fontsize=9, fontweight='bold')
for b, n in zip(bars, names):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01, f"{SUMMARY[n]['unseen']:.3f}",
            ha='center', fontsize=9)
ax.set_ylabel('Mean UNSEEN mDice'); ax.set_ylim(0, 1)
ax.set_title('Unseen-data performance: prompt-free (blue) vs oracle-prompted (red)')
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=20, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

print(f'{"model":<26} {"seen":>7} {"unseen":>8} {"gap":>8}   tier')
print('-' * 62)
for n in names:
    s = SUMMARY[n]
    flag = '   <- negative gap = oracle tell' if s['gap'] < 0 else ''
    print(f'{n:<26} {s["seen"]:>7.3f} {s["unseen"]:>8.3f} {s["gap"]:>+8.3f}   {s["tier"]}{flag}')

## 7. Findings — the honest read

**Fair comparison (prompt-free models):**
- **SAM-ViT-H + LoRA generalizes best to unseen data (0.792)** — better than the from-scratch U-Net
  specialist (0.752) and with the smallest seen→unseen drop (+0.096 vs +0.157), while training only
  **830K params (0.4% of the model)** vs U-Net's 24.4M.
- **U-Net** fits *seen* data best (0.909) but drops the most on unseen — the classic specialist pattern.
- **MedSAM-ViT-B + LoRA is weakest (0.660 unseen)** — but this is **confounded**: MedSAM is ViT-B while
  SAM is ViT-H, so this is partly a capacity gap, not a verdict on medical pretraining. *To isolate it,
  train SAM ViT-B + LoRA.*

**Oracle tier (vanilla SAM/MedSAM, GT-box prompted):**
- Given the box, they trace polyps at **0.85–0.90 Dice** — but the **negative generalization gap** shows
  this measures *boundary tracing given known location*, not detection. It is an **upper bound**, not a win.
- The Section-5 gallery makes it concrete: the biggest vanilla "wins" are exactly the images where the
  fine-tuned model *couldn't find* the polyp. Fine-tuning's job is that detection — which the oracle box
  removes.

**Bottom line:** fine-tuning a foundation model with a tiny LoRA adapter is what lets it *detect* polyps
and *generalize* across datasets. Vanilla SAM is an excellent boundary tracer **only if something else
localizes the polyp first** — which, in a real deployment, nothing does.

**Limitations to state in the write-up:** oracle-prompt asymmetry (above); single seed (42) — the
SAM-LoRA vs U-Net edge is modest and wants 2–3 seeds; MedSAM backbone confound (ViT-B vs ViT-H).